# Lab 8 — Benchmarking, Evals, and Leaderboards

Lab 7 built a single agent and watched what `temperature` did to it. But once
you can *build* models and agents, the next question is unavoidable: **how do
you know if one is any good — and better than another?**

This lab is about *measuring*. Three ideas, in increasing order of slipperiness:

1. **Evals** — score one model on a task with a known answer. We use **GSM8K**
   (grade-school math). The lesson: an eval is `task + metric + protocol`, and
   *the grading harness is as much the eval as the model is.*
2. **Benchmarking** — what goes wrong when you compare models on a fixed
   benchmark: saturation, contamination, protocol sensitivity.
3. **Leaderboards** — ranking *many* models against each other. For verifiable
   benchmarks you rank by **score**; for open-ended tasks with **no** gold
   answer you rank by **pairwise preference → Elo** / Bradley–Terry (Chatbot
   Arena). We look at real leaderboards of both kinds and build Elo from scratch.

We build the eval harness from scratch as a tiny LangGraph (`solve → grade`),
in the same spirit as Lab 7's two-node ReAct loop — so the eval is visible in
the graph, not hidden in a library.

## Prerequisites

Same setup as Labs 5–7:

- `pip install -e '~/GitHub/cmbagent_lg'` (plain install is enough)
- `GOOGLE_API_KEY` set in `~/GitHub/cmbagent_lg/.env` (or as an env var / Colab secret)
- (optional) `LANGFUSE_*` in the same `.env` for tracing
- `numpy` and `matplotlib` for the metric plots and the Elo simulation

Pick the **Python 3.12 (cmbagent_lg)** kernel.

In [ ]:
# Robust key loading: local .env (Labs 5-7), else Colab secret, else env var.
import os

try:
    from dotenv import load_dotenv
    load_dotenv('/Users/boris/GitHub/cmbagent_lg/.env', override=True)
except Exception:
    pass

if not os.environ.get('GOOGLE_API_KEY'):
    try:  # Colab
        from google.colab import userdata
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        pass

assert os.environ.get('GOOGLE_API_KEY'), 'set GOOGLE_API_KEY (env var, .env, or Colab secret)'

# Langfuse is optional — attach the handler if its keys are present.
callbacks, handler = [], None
try:
    from cmbagent_lg.tracing import langfuse_handler
    handler = langfuse_handler()
    callbacks = [handler]
    print('env OK — Langfuse tracing attached')
except Exception as e:
    print(f'env OK — running without Langfuse ({e})')

## 1. What is an eval?

An eval has **three** parts, and changing *any* of them changes the number:

| Part | GSM8K example | Why it matters |
| --- | --- | --- |
| **Task** | grade-school math word problems | defines *what* you measure |
| **Metric** | accuracy (extracted answer == gold) | defines *how* you score |
| **Protocol** | zero-shot, chain-of-thought, T=0, k=1 | same model, different protocol → up to 20+ points |

A benchmark number with no protocol attached is meaningless. Keep that in mind
every time you read "model X scores Y% on Z".

### A *verifiable* task: GSM8K

GSM8K (Cobbe et al., 2021) is the easy case for evals: every problem has a
**single integer answer**, so grading is — in principle — a one-liner. We
embed five canonical problems so the lab runs offline.

In [ ]:
# (question, gold_integer) — the canonical first five GSM8K problems.
GSM8K = [
    ("Natalia sold clips to 48 of her friends in April, and then she sold half "
     "as many clips in May. How many clips did she sell altogether in April and May?", 72),
    ("Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes "
     "of babysitting. How much did she earn?", 10),
    ("Betty is saving money for a new wallet which costs $100. Betty has only half "
     "of the money she needs. Her parents decided to give her $15, and her "
     "grandparents twice as much as her parents. How much more money does Betty "
     "need to buy the wallet?", 5),
    ("James writes a 3-page letter to 2 different friends twice a week. How many "
     "pages does he write a year?", 624),
    ("A robe takes 2 bolts of blue fiber and half that much white fiber. How many "
     "bolts in total does it take?", 3),
]
for q, g in GSM8K:
    print(f"[{g:>4}] {q[:70]}...")

### The grader *is* the eval

GSM8K's gold answers live after a `####` marker. So the model is asked to end
with `#### <number>`, and we extract that number. But what if the model gets the
right answer and forgets the format? We grade **two ways** to expose this:

- **strict** — the number must come after the final `####` (the GSM8K convention).
- **lenient** — just take the last number anywhere in the response.

The *same generation* can pass one and fail the other. The gap between them is
"right reasoning, wrong format" — points the harness gives or takes away. This
is the single most important eval lesson: **a benchmark score is partly a
property of your parsing code.**

In [ ]:
import re
_NUM = re.compile(r"-?\d[\d,]*")

def extract_strict(text: str):
    """GSM8K convention: the number after the final '####'."""
    if "####" not in text:
        return None
    m = _NUM.search(text.rsplit("####", 1)[-1])
    return int(m.group().replace(",", "")) if m else None

def extract_lenient(text: str):
    """Just the last number anywhere in the response."""
    nums = _NUM.findall(text)
    return int(nums[-1].replace(",", "")) if nums else None

# quick self-test — note row 3: right number, no '####' -> strict fails, lenient ok
for t in ["The total is 72.\n#### 72", "so she earned 10 dollars.\n#### 10",
          "I'm confident the answer is 3 bolts."]:
    print(f"strict={str(extract_strict(t)):>5}  lenient={str(extract_lenient(t)):>5}   <- {t!r}")

## 2. A from-scratch eval harness: `solve → grade`

Like Lab 7, we build the harness as an explicit LangGraph instead of calling a
library, so the structure is visible:

```
START ──▶ solve ──▶ grade ──▶ END
         (model)   (deterministic Python)
```

- **`solve`** asks the model to reason and answer.
- **`grade`** extracts the answer (both ways) — *no model call*, pure code.

Putting `grade` in the graph on purpose makes the point that the grader is a
first-class part of the eval, not an afterthought.

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

SYSTEM_PROMPT = ("You are a careful grade-school math tutor. Think step by step, "
                 "then on the final line write the answer as `#### <number>` and nothing else.")

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    strict: int | None
    lenient: int | None

def make_graph(temperature: float):
    model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",
                                   temperature=temperature, thinking_level="low")

    def solve(state):  # the model node
        return {"messages": [model.invoke(state["messages"])]}

    def grade(state):  # deterministic node — the grader is part of the eval
        text = state["messages"][-1].content
        if isinstance(text, list):  # gemini can return content parts
            text = " ".join(str(p) for p in text)
        return {"strict": extract_strict(text), "lenient": extract_lenient(text)}

    g = StateGraph(State)
    g.add_node("solve", solve)
    g.add_node("grade", grade)
    g.add_edge(START, "solve")
    g.add_edge("solve", "grade")
    g.add_edge("grade", END)
    # NB: no compile(cache=...) — every sample must re-run, else repeated
    # samples (k>1) would collapse to one cached result and kill pass@k/maj@k.
    return g.compile()

graph0 = make_graph(temperature=0.0)
print("graph compiled:", [n for n in graph0.get_graph().nodes])

In [ ]:
def run_once(graph, question, gold):
    final = graph.invoke({"messages": [SystemMessage(content=SYSTEM_PROMPT),
                                       HumanMessage(content=question)]},
                         config={"callbacks": callbacks})
    return {"strict": final.get("strict"), "lenient": final.get("lenient")}

# Deterministic pass (T=0): plain accuracy + the harness gap.
strict_ok = lenient_ok = 0
print(f"{'gold':>5}  {'strict':>7} {'lenient':>8}")
for q, gold in GSM8K:
    r = run_once(graph0, q, gold)
    strict_ok  += int(r["strict"]  == gold)
    lenient_ok += int(r["lenient"] == gold)
    print(f"{gold:>5}  {str(r['strict']):>7} {str(r['lenient']):>8}")

n = len(GSM8K)
print(f"\nstrict  accuracy: {strict_ok}/{n} = {strict_ok/n:.0%}")
print(f"lenient accuracy: {lenient_ok}/{n} = {lenient_ok/n:.0%}")
if strict_ok != lenient_ok:
    print(f"-> {lenient_ok-strict_ok} answer(s) right but strict harness scored wrong (format, not reasoning)")

On these five easy problems a strong model usually scores 100% and follows the
format, so strict == lenient and the gap is zero. That's the *boring baseline* —
and exactly why benchmarks need harder, format-stressing inputs to be
discriminative. We'll make the gap appear in the exercises by weakening the
format instruction or pulling real GSM8K test items.

## 3. Sampling more than once: pass@1, pass@k, maj@k

Real models are stochastic at `temperature > 0`. If we draw **k** samples per
question, three different metrics fall out — and they answer different questions:

- **pass@1** — is the **first** sample correct? Plain one-shot accuracy.
  Independent of k.
- **pass@k** — is **any** of the k samples correct? An **optimistic ceiling**:
  it assumes a *perfect verifier* that can pick the right one out of k. Only
  achievable in practice if you actually have such a verifier (a unit test, a
  proof checker). For free-form QA you usually don't, so pass@k *overstates*
  real accuracy.
- **maj@k** — take the **majority vote** of the k answers, then grade it. This
  is **self-consistency** (Wang et al., 2022): no verifier needed, you trust the
  consensus. It denoises a model that is right *on average* but wobbly
  per-sample.

Typically **pass@1 ≤ maj@k ≤ pass@k**.

Worked example, k=5, extracted answers, gold=72:
- `[72, 72, 71, 72, 70]` → pass@1 ✓, pass@k ✓, maj@k ✓ (72 wins 3–1–1).
- `[71, 72, 71, 70, 71]` → pass@1 ✗ (first is 71), pass@k ✓ (a 72 exists),
  **maj@k ✗** (71 wins the vote → consensus is *wrong*). This is the case that
  separates the optimistic ceiling from what voting actually recovers.

In [ ]:
from collections import Counter

def eval_question(graph, question, gold, k):
    preds = [run_once(graph, question, gold)["lenient"] for _ in range(k)]
    votes = Counter(p for p in preds if p is not None)
    vote = votes.most_common(1)[0][0] if votes else None
    return {
        "preds": preds,
        "pass1": preds[0] == gold,
        "passk": any(p == gold for p in preds),
        "majk":  vote == gold,
        "vote":  vote,
    }

K, T = 5, 0.8
graphT = make_graph(temperature=T)
agg = {"pass1": 0, "passk": 0, "majk": 0}
print(f"k={K}, T={T}\n{'gold':>5}  {'vote':>5}  samples")
for q, gold in GSM8K:
    r = eval_question(graphT, q, gold, K)
    for m in agg: agg[m] += int(r[m])
    flags = "".join("." if p == gold else "x" for p in r["preds"])
    print(f"{gold:>5}  {str(r['vote']):>5}  [{flags}]")

n = len(GSM8K)
print(f"\npass@1  (first sample)        : {agg['pass1']}/{n} = {agg['pass1']/n:.0%}")
print(f"maj@{K}   (self-consistency)     : {agg['majk']}/{n} = {agg['majk']/n:.0%}")
print(f"pass@{K}  (any correct, oracle)  : {agg['passk']}/{n} = {agg['passk']/n:.0%}")

**Temperature is the knob that makes these diverge.** At `T=0` the k samples are
near-identical, so `maj@k ≈ pass@1` and `pass@k` barely moves. Raise `T` and the
samples spread out: maj@k can climb above pass@1 (voting cleans up noise) and
pass@k rises fastest (diversity helps *if* you can verify). Try re-running the
cell above with `T = 0.0` and `T = 1.0`.

## 4. Benchmarking pitfalls

A single accuracy number on a fixed benchmark hides a lot. The four that bite
hardest:

| Pitfall | What happens | Why it matters |
| --- | --- | --- |
| **Saturation** | frontier models score ~95%+ on GSM8K | the benchmark stops discriminating; you need harder ones (MATH, GSM-Hard) |
| **Contamination** | the test set leaked into pretraining | high score = memorisation, not reasoning |
| **Protocol sensitivity** | zero/few-shot, CoT, prompt format | same model swings 20+ points; numbers aren't comparable across protocols |
| **Single scalar** | one accuracy number | hides *which* problems fail; no error analysis, no partial credit |

The sharpest demonstration of contamination is **GSM-Symbolic** (Mirzadeh et
al., 2024): regenerate the *same* GSM8K problems with different names and
numbers, and accuracy **drops** — evidence that models partly pattern-match
memorised instances rather than reason. The cell below mimics that idea: we
swap the surface details of a problem and check the model still gets it.

In [ ]:
# A tiny "symbolic" perturbation: same structure, new numbers/names.
# Original Natalia problem (gold 72) -> perturbed (gold 60).
PERTURBED = ("Marcus sold pencils to 40 of his classmates on Monday, and then he "
             "sold half as many pencils on Tuesday. How many pencils did he sell "
             "altogether on Monday and Tuesday?", 60)

q, gold = PERTURBED
r = run_once(graph0, q, gold)
print(f"perturbed problem, gold={gold}: lenient={r['lenient']}  -> "
      f"{'correct' if r['lenient']==gold else 'WRONG'}")
print("\n(Real GSM-Symbolic runs hundreds of such perturbations and reports the "
      "accuracy *distribution* — a drop vs the memorised originals signals "
      "contamination rather than reasoning.)")

## 5. Leaderboards

A **leaderboard** ranks many models against each other. There are two flavours,
and which one you use is decided by the *task*, not by whether you happen to
have a leaderboard:

**(a) Verifiable benchmarks → rank by score.** When the task *has* a gold answer
(math, code, multiple-choice), you run every model through an eval like the one
in Sections 1–4 and sort by accuracy. **Most public leaderboards are this kind:**

- [Artificial Analysis](https://artificialanalysis.ai/) — aggregates dozens of
  benchmarks into combined *intelligence / speed / price* leaderboards across
  providers.
- [TPBench](https://tpbench.org/) — the *Theoretical Physics Benchmark*:
  research-level physics problems with checkable answers (right up our street).
- [Terminal-Bench](https://www.tbench.ai/) — agents solving real tasks in a
  terminal, graded by **automated tests** (verifiable, but *agentic* — much
  closer to what this course is about than a single-answer quiz).

So leaderboards are *not* a no-gold-only thing — accuracy-ranked leaderboards on
verifiable benchmarks are the common case.

**(b) Open-ended tasks → rank by preference.** "Write a poem", "explain this
bug", "summarise this paper" have **no gold answer** — you can't exact-match. So
instead you collect **pairwise preferences**: show two models' answers to the
same prompt, ask a judge (human or LLM) *which is better*, and aggregate the
wins into a ranking. This is how **Chatbot Arena / LMArena** ranks models, and
it's the case the rest of this section builds.

The aggregation tool is **Elo** (Arpad Elo, chess). Each model has a rating `R`;
the rating gap predicts win probability via a logistic curve:

$$P(A \text{ beats } B) = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$

A 400-point gap → ~10:1 odds (≈91%). After each battle you nudge ratings toward
the actual outcome `S ∈ {1, 0.5, 0}`:

$$R_A \leftarrow R_A + K\,(S_A - P(A \text{ beats } B))$$

`K` is the step size: beating someone you were *expected* to beat moves you
little; an upset moves you a lot. Only **differences** matter — the absolute
scale is a convention (Arena anchors it).

In [ ]:
def expected_score(ra, rb):
    """Logistic win probability of A over B given Elo ratings."""
    return 1.0 / (1.0 + 10 ** ((rb - ra) / 400.0))

def elo_update(ra, rb, sa, K=32):
    """Return updated (ra, rb) after a game where A scored sa in {1, .5, 0}."""
    ea = expected_score(ra, rb)
    ra += K * (sa - ea)
    rb += K * ((1 - sa) - (1 - ea))
    return ra, rb

# sanity: equal ratings -> 50/50; +400 -> ~0.91
print("P(equal)      =", round(expected_score(1500, 1500), 3))
print("P(+400 ahead) =", round(expected_score(1900, 1500), 3))

### Simulating a leaderboard

Let's give four "models" hidden *true* skills, simulate random pairwise battles
where the better model wins with the Elo-implied probability, and run online Elo
starting everyone at 1500. The recovered ratings should sort into the true
order — that's the leaderboard emerging from noisy pairwise votes.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

models = ["tiny", "small", "medium", "large"]
true_skill = {"tiny": 1300, "small": 1500, "medium": 1650, "large": 1800}

rating = {m: 1500.0 for m in models}      # everyone starts equal
history = {m: [rating[m]] for m in models}

N_BATTLES = 4000
for _ in range(N_BATTLES):
    a, b = rng.choice(models, size=2, replace=False)
    p_a_true = expected_score(true_skill[a], true_skill[b])   # ground-truth win prob
    sa = 1.0 if rng.random() < p_a_true else 0.0              # sampled outcome
    rating[a], rating[b] = elo_update(rating[a], rating[b], sa)
    for m in models:
        history[m].append(rating[m])

print(f"{'model':>8}  {'true':>6}  {'recovered Elo':>13}")
for m in sorted(models, key=lambda x: -rating[x]):
    print(f"{m:>8}  {true_skill[m]:>6}  {rating[m]:>13.0f}")

In [ ]:
# Watch the ratings converge as battles accumulate.
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    for m in models:
        plt.plot(history[m], label=m)
        plt.axhline(true_skill[m], ls=":", color="gray", lw=0.7)
    plt.xlabel("battles"); plt.ylabel("Elo rating")
    plt.title("Online Elo converging toward hidden true skill (dotted)")
    plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)

### Caveats worth a slide

- **Order / non-stationarity.** Textbook sequential Elo depends on game order and
  assumes fixed skill. Chatbot Arena actually fits the **Bradley–Terry** model — a
  *static* maximum-likelihood fit over **all** comparisons at once — and reports
  it *as* "Elo". It's more stable and gives confidence intervals. So "Arena Elo"
  ≠ the running update above; the update is the intuition, BT is the production
  estimator.
- **Intransitivity.** A > B > C > A can happen; a single number hides it.
- **Prompt distribution.** The ranking only reflects the prompts people actually
  submitted.

### LLM-as-judge (and its biases)

Humans are slow, so leaderboards increasingly use an **LLM as the judge** of the
pairwise comparison. Cheap and scalable — but the judge has biases. The biggest
is **position bias**: a tendency to prefer whichever answer is shown *first*,
regardless of content. The standard mitigation is to **run both orderings** and
only count a win if it survives the swap. The cell below demonstrates the
check.

In [ ]:
judge = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0,
                               thinking_level="low")

PROMPT = "Explain in one sentence why the sky is blue."
ANS_A = ("Sunlight scatters off air molecules, and shorter (blue) wavelengths "
         "scatter much more than longer ones (Rayleigh scattering), so the sky "
         "looks blue.")
ANS_B = "Because of the air and the sun and the light going around."

def judge_pair(prompt, first, second):
    msg = (f"Question: {prompt}\n\nAnswer 1:\n{first}\n\nAnswer 2:\n{second}\n\n"
           "Which answer is better? Reply with exactly '1' or '2'.")
    out = judge.invoke(msg).content
    out = out if isinstance(out, str) else " ".join(map(str, out))
    return "1" if "1" in out[:3] else "2"

v1 = judge_pair(PROMPT, ANS_A, ANS_B)   # A shown first
v2 = judge_pair(PROMPT, ANS_B, ANS_A)   # B shown first
winner_1 = "A" if v1 == "1" else "B"
winner_2 = "A" if v2 == "2" else "B"     # A is now in slot 2
print(f"order (A,B): picked slot {v1} -> {winner_1}")
print(f"order (B,A): picked slot {v2} -> {winner_2}")
print("=> consistent winner:" , winner_1 if winner_1 == winner_2 else
      "NONE — position bias detected (winner flipped with order)")

Other judge biases to keep in mind: **length bias** (preferring longer/more
verbose answers), **self-preference** (an LLM favouring answers from its own
model family), and **style over substance** (confident formatting beating
correct content). A robust judge protocol swaps order, may average several
judges, and ideally calibrates against human labels.

## 6. Cost vs score: the Pareto frontier

A leaderboard sorted by score alone hides half the decision. In practice you
never just want "the highest score" — you want the best score **you can afford**,
fast enough. So evals are increasingly reported as a **trade-off plot**: score on
one axis, **cost** (or latency) on the other, one point per model. This is
exactly what [Artificial Analysis](https://artificialanalysis.ai/) shows —
*intelligence vs price*, *intelligence vs speed*.

The useful object on that plot is the **Pareto frontier**: the set of models that
are **not dominated** — i.e. no other model is *both* cheaper *and* better. A
model below the frontier is strictly worse than something on it (there exists a
model that beats it on score *and* costs less), so it's never the rational pick.
Models *on* the frontier are the real choices; *where* you sit on it is set by
your budget and latency constraints, not by the benchmark.

(This is also the right lens for the Part I exam: judge a GRPO change on a
**score-vs-compute** frontier — a small, cheap, *explained* gain can dominate a
large but expensive one.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A synthetic leaderboard: (name, cost in $/Mtok, score).
data = [("nano", 0.3, 40), ("A", 0.5, 55), ("B", 1.0, 50), ("H", 1.5, 58),
        ("C", 2.0, 70), ("D", 3.0, 64), ("E", 5.0, 80), ("F", 8.0, 72),
        ("G", 12.0, 86)]

# Pareto frontier: maximise score, minimise cost. Sweep by ascending cost and
# keep a model only if it beats the best score seen so far (cheaper => must be better).
frontier, best = [], -1.0
for name, cost, score in sorted(data, key=lambda x: (x[1], -x[2])):
    if score > best:
        frontier.append((name, cost, score)); best = score
front_names = {f[0] for f in frontier}

print("on the frontier:", [f[0] for f in frontier])
print("dominated      :", [d[0] for d in data if d[0] not in front_names])

try:
    plt.figure(figsize=(7, 4.5))
    for name, cost, score in data:
        on = name in front_names
        plt.scatter(cost, score, s=90, c=("tab:green" if on else "tab:red"), zorder=3)
        plt.annotate(name, (cost, score), textcoords="offset points", xytext=(6, 4))
    fx = [f[1] for f in frontier]; fy = [f[2] for f in frontier]
    plt.plot(fx, fy, "--", c="tab:green", label="Pareto frontier")
    plt.xlabel("cost  ($ / Mtok)  -> cheaper is better (left)")
    plt.ylabel("score  -> better is up")
    plt.title("Cost vs score: green = Pareto-optimal, red = dominated")
    plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)

Read the plot: a **red** point always has a **green** point up-and-to-the-left
of it (better *and* cheaper) — that's domination. The dashed line is the
frontier: the most score available at each price. "Which model is best?" has no
single answer — it's "which frontier point matches my budget?"

## 7. The eval taxonomy

Three regimes, three toolkits — pick by whether a *gold answer* exists:

| Regime | Ground truth | How you score |
| --- | --- | --- |
| **Verifiable** (GSM8K, code, math) | single correct answer | exact-match accuracy, pass@k, maj@k |
| **Open-ended** (chat, writing) | none | pairwise preference → Elo / Bradley–Terry |
| **Rubric / graded** (essays, summaries) | reference + rubric | LLM-as-judge score against the rubric |

**Leaderboards sit on top of all three rows**, not just the open-ended one: a
verifiable benchmark gives a *score* leaderboard (Artificial Analysis, TPBench,
Terminal-Bench), open-ended tasks give an *Elo* leaderboard (Chatbot Arena), and
rubric grading gives a *judge-score* leaderboard. Elo is the tool for the no-gold
case specifically — not a synonym for "leaderboard".

Same theme throughout: **the metric and the protocol are part of the eval**, and
a leaderboard number is only as trustworthy as the eval or judging process
behind it.